In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Optimized code
# 1. Compute mask only once, reusing column references to avoid repeated attribute lookups
# 2. Select only needed columns before merge to reduce memory copy
# 3. Replace Python‐level g1/g2 with vectorized .isin and named aggregations

date1 = pd.Timestamp("1994-01-01")
date2 = pd.Timestamp("1995-01-01")

df = lineitem
lrc = df.L_RECEIPTDATE
lcd = df.L_COMMITDATE
lsd = df.L_SHIPDATE
mode = df.L_SHIPMODE

sel = (
    (lrc >= date1) & (lrc < date2)
    & (lcd < lrc) & (lcd < date2)
    & (lsd < lcd) & (lsd < date2)
    & mode.isin(["MAIL", "SHIP"])
)

# Keep only the join keys and grouping key
flineitem = df.loc[sel, ["L_ORDERKEY", "L_SHIPMODE"]]

# Merge only the columns we need from orders
jn = flineitem.merge(
    orders[["O_ORDERKEY", "O_ORDERPRIORITY"]],
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY"
)

# Vectorized flag for high-priority orders
jn["is_high"] = jn["O_ORDERPRIORITY"].isin(["1-URGENT", "2-HIGH"])

# Group once, computing sum(is_high) and size per ship mode
total = (
    jn.groupby("L_SHIPMODE", as_index=False)["is_high"]
      .agg(g1="sum", cnt="size")
)
# g2 = count of non-high-priority
total["g2"] = total["cnt"] - total["g1"]

# Final columns
total = total[["L_SHIPMODE", "g1", "g2"]]